# Optimizing LibreYOLO for Qualcomm® Snapdragon Devices Using QAIRT

This notebook walks through the full pipeline to optimize a **LibreYOLO** object detection model for efficient inference on Qualcomm® Snapdragon devices using the **Qualcomm AI Runtime (QAIRT) SDK**.

The pipeline covers:
1. **Image preprocessing** — resize and normalize images for model input.
2. **Calibration dataset preparation** — generate representative `.raw` samples for INT8 quantization.
3. **Model export** — load a pre-trained LibreYOLOXs checkpoint and export it to ONNX.
4. **DLC conversion** — convert the ONNX model to QAIRT's Deep Learning Container (DLC) format using `qairt-converter`.
5. **INT8 quantization** — apply post-training quantization for faster, more efficient inference using `qairt-quantizer`.

We begin by importing the necessary libraries.

In [ ]:
# Import necessary libraries.
import cv2
import glob
import os
import onnx
import random
import torch
import uuid

import numpy as np
import pandas as pd

from libreyolo.models.yolox.nn import LibreYOLOXModel
from onnx import helper, TensorProto
from typing import Tuple

## Preprocessing and Calibration Data

Before converting or quantizing the model, two things are needed:

- A **preprocessing function** to transform raw images into the tensor format expected by LibreYOLO (letterbox-resized to 640×640, BGR color, 0–255 float32, CHW layout).
- A **representative calibration dataset** in `.raw` binary format. QAIRT's `qairt-quantizer` uses this dataset during post-training quantization to compute per-layer scale factors for INT8 weights and activations.

The cell below defines the `letterbox()` and `preprocess()` helper functions used throughout this pipeline.

In [ ]:
def letterbox(
    image: np.ndarray, target: int = 640, pad_value: int = 114
) -> Tuple:
    """Resize image to (target×target) with top-left letterbox padding.

    YOLOX preprocessing: image is placed at top-left (0, 0); padding fills
    the right and bottom edges. Color format stays BGR, values stay 0–255.

    Args:
        image (np.ndarray): Input BGR image as NumPy array.
        target (int): Target image size (square).
        pad_value (int): Padding pixel value (default 114, YOLOX convention).

    Returns:
        padded (np.ndarray): Padded image, float32 HWC BGR, 0-255 range.
        ratio (float): Scale factor applied to original dimensions.
    """
    h, w = image.shape[:2]
    ratio = min(target / h, target / w)
    new_w, new_h = int(w * ratio), int(h * ratio)

    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    padded = np.full((target, target, 3), pad_value, dtype=np.float32)
    padded[:new_h, :new_w] = resized

    return padded, ratio


def preprocess(original_image: np.ndarray, size: int = 640) -> Tuple:
    """
    Preprocess the input image for model inference.

    YOLOX expects: BGR, 0-255 float32, CHW, top-left letterbox.
    No /255 normalisation and no BGR→RGB conversion.

    Args:
        original_image (np.ndarray): Input BGR image as NumPy array.
        size (int): Target image size.

    Returns:
        np.ndarray: Preprocessed float32 image, CHW layout, 0-255 range.
        float: Scale ratio applied to original dimensions.
    """
    padded, ratio = letterbox(original_image, size)
    return padded.transpose(2, 0, 1), ratio

### Preparing the Calibration Dataset

QAIRT's quantization tool (`qairt-quantizer`) requires input samples in `.raw` format — flat binary files containing `float32` pixel values with shape `(C, H, W)`.

The code below:
1. Downloads the **COCO 2017 validation set** (~777 MB) and **COCO 2017 validation annotation** (~241 MB)
2. Randomly samples **1000 images** for calibration and **100 images** for validation (with a fixed seed for reproducibility).
3. Preprocesses each image using `preprocess()` and saves it as a `.raw` file.
4. Generates a `filenames.txt` manifest listing all `.raw` files — required by `qairt-quantizer`.
5. Generates a `metadata.csv` for data analysis.

In [ ]:
if not os.path.exists("dataset"):
    !bash -c 'curl -L -o val2017.zip http://images.cocodataset.org/zips/val2017.zip'
    !bash -c 'unzip val2017.zip -d dataset'
    !bash -c 'rm val2017.zip'

    !bash -c 'curl -L -o annotations_trainval2017.zip http://images.cocodataset.org/annotations/annotations_trainval2017.zip'
    !bash -c 'unzip annotations_trainval2017.zip -d dataset'
    !bash -c 'rm annotations_trainval2017.zip'

if not os.path.exists("qairt/calib"):
    os.makedirs("qairt/calib", exist_ok=True)
    os.makedirs("qairt/val", exist_ok=True)

    random.seed(42)
    SAMPLE_SIZE = 1000

    filenames = glob.glob(f"dataset/**/*.jpg", recursive=True)
    random.shuffle(filenames)

    for filename in filenames[:SAMPLE_SIZE]:
        original_image = cv2.imread(filename)
        processed_image, _ = preprocess(original_image)
        processed_image = processed_image.reshape(1, 3, 640, 640)
        processed_image.tofile(f"qairt/calib/{uuid.uuid4()}.raw")

    !bash -c 'find qairt/calib -name "*.raw" > qairt/calib/filenames.txt'

    metadata = []
    for i, filename in enumerate(
        filenames[SAMPLE_SIZE:SAMPLE_SIZE + int(SAMPLE_SIZE * .1)]
    ):
        original_image = cv2.imread(filename)
        processed_image, ratio = preprocess(original_image)
        processed_image = processed_image.reshape(1, 3, 640, 640)
        processed_image.tofile(raw_path := f"qairt/val/{uuid.uuid4()}.raw")

        metadata.append({
            "raw_path": str(raw_path),
            "image_path": str(filename),
            "image_id": i + 1,
            "orig_w": original_image.shape[1],
            "orig_h": original_image.shape[0],
            "ratio": ratio,
        })

    metadata_df = pd.DataFrame(metadata)
    metadata_df.to_csv("qairt/val/metadata.csv", index=False)

    !bash -c 'find qairt/val -name "*.raw" > qairt/val/filenames.txt'

### Loading LibreYOLO and Exporting to ONNX

QAIRT does not consume LibreYOLO models in PyTorch format directly. The model must first be exported to the **ONNX (Open Neural Network Exchange)** format, which QAIRT can then parse and convert to its DLC format.

The code below:
1. Downloads the pre-trained **LibreYOLOXs** checkpoint from Hugging Face.
2. Loads it using the `LibreYOLO` wrapper and sets it to evaluation mode.
3. Exports it to ONNX using `torch.onnx.export()` with `opset_version=13`.
4. Applies **ONNX graph surgery** to eliminate all mixed-range intermediate tensors and produce two independent output branches.

The model decodes the grid offsets inside the graph and returns two outputs:

| Output   | Shape          | Range  | Description                              |
|----------|----------------|--------|------------------------------------------|
| `bboxes` | `[1, 8400, 4]` | 0–640  | Decoded bbox coords `[cx, cy, w, h]`     |
| `scores` | `[1, 8400, 81]`| 0–1    | Objectness + 80 class probabilities      |

where `8400 = 80×80 + 40×40 + 20×20` anchor-free grid cells.

The bbox format is `[cx, cy, w, h]` in the resized 640×640 input coordinate space. When using the model, confidence filtering, `cxcywh → xyxy` conversion, scaling back to the original image, and NMS are still required.

In [ ]:
os.makedirs("../models", exist_ok=True)
os.makedirs("../models/qairt", exist_ok=True)

if not os.path.exists("../models/LibreYOLOXs.pt"):
    !bash -c 'curl -L -o ../models/LibreYOLOXs.pt https://huggingface.co/LibreYOLO/LibreYOLOXs/resolve/main/LibreYOLOXs.pt?download=true'

checkpoint = torch.load(
    "../models/LibreYOLOXs.pt",
    map_location="cpu"
)

model = LibreYOLOXModel(config="s", nb_classes=80)
model.load_state_dict(checkpoint["model"], strict=True)

model.eval()
model.head.export = True


def _split_onnx_for_dsp(path):
    """Restructure the ONNX graph to eliminate mixed-range intermediate tensors.

    The original YOLOX export produces per-scale Concat nodes that mix decoded
    bbox coordinates (range 0-640) with sigmoid scores (range 0-1) in a single
    [1, n, 85] tensor.  With INT8 per-tensor quantization, the scale is set to
    cover 0-640, which maps all 0-1 score values to INT8 zero on DSP.

    This function removes those mixed-range Concat nodes and wires the already-
    existing [1, n, 81] score tensors directly into a separate cross-scale
    Concat, producing two independent output branches with their own INT8 scales:

        bboxes  [1, 8400, 4]  -- range 0-640   -> INT8 scale ~2.5 / step
        scores  [1, 8400, 81] -- range 0-1      -> INT8 scale ~0.004 / step

    Every new node is given an explicit name so that SNPE/QAIRT DLC layer
    names are predictable (e.g. setOutputLayers({"bboxes", "scores"})).

    The function is idempotent: if no mixed-range nodes are found, it prints a
    message and returns without modifying the file.
    """
    model = onnx.load(path)
    model = onnx.shape_inference.infer_shapes(model)
    graph = model.graph

    shape_map = {
        vi.name: [d.dim_value for d in vi.type.tensor_type.shape.dim]
        for vi in list(graph.value_info) + list(graph.input) + list(graph.output)
        if vi.type.HasField("tensor_type") and vi.type.tensor_type.shape
    }

    SCALE_NS = {6400, 1600, 400}
    per_scale, final_cat = [], None

    for node in graph.node:
        if node.op_type != "Concat":
            continue
        s = shape_map.get(node.output[0], [])
        ax = next((a.i for a in node.attribute if a.name == "axis"), None)
        if len(s) == 3 and s[0] == 1 and s[2] == 85 and s[1] in SCALE_NS and ax == -1:
            per_scale.append(node)
        elif s == [1, 8400, 85] and ax == 1:
            final_cat = node

    if not per_scale:
        print("ONNX graph already restructured, skipping.")
        return

    assert len(per_scale) == 3 and final_cat is not None, \
        "Unexpected graph structure — cannot apply surgery."

    old_to_new, new_nodes = {}, []

    for node in per_scale:
        n_k = shape_map[node.output[0]][1]
        inps = list(node.input)
        bbox_inps  = [i for i in inps if shape_map.get(i, [])[-1:] == [2]]
        score_inps = [i for i in inps if shape_map.get(i, [])[-1:] == [81]]
        bbox_name = f"_split_bboxes_{n_k}"
        new_nodes.append(helper.make_node(
            "Concat", inputs=bbox_inps, outputs=[bbox_name],
            name=bbox_name, axis=-1
        ))
        old_to_new[node.output[0]] = (bbox_name, score_inps[0])

    all_bboxes = [old_to_new[i][0] for i in final_cat.input]
    all_scores = [old_to_new[i][1] for i in final_cat.input]
    new_nodes.append(helper.make_node(
        "Concat", inputs=all_bboxes, outputs=["bboxes"],
        name="bboxes", axis=1
    ))
    new_nodes.append(helper.make_node(
        "Concat", inputs=all_scores, outputs=["scores"],
        name="scores", axis=1
    ))

    final_tensor = final_cat.output[0]
    remove_ids = {id(n) for n in per_scale} | {id(final_cat)}
    for node in graph.node:
        if final_tensor in list(node.input):
            remove_ids.add(id(node))

    new_graph_nodes = [n for n in graph.node if id(n) not in remove_ids] + new_nodes
    del graph.node[:]
    graph.node.extend(new_graph_nodes)

    while graph.output:
        graph.output.pop()
    graph.output.extend([
        helper.make_tensor_value_info("bboxes", TensorProto.FLOAT, [1, 8400, 4]),
        helper.make_tensor_value_info("scores", TensorProto.FLOAT, [1, 8400, 81]),
    ])

    model = onnx.shape_inference.infer_shapes(model)
    onnx.checker.check_model(model)
    onnx.save(model, path)
    print("ONNX graph restructured: bboxes and scores are now separate branches.")

dummy = torch.randn(1, 3, 640, 640)
onnx_path = "../models/LibreYOLOXs.onnx"

if not os.path.exists(onnx_path):
    torch.onnx.export(
        model,
        dummy,
        onnx_path,
        opset_version=13,
        dynamo=False,
        do_constant_folding=True,
        input_names=["images"],
        output_names=["detections"]
    )

    print(f"Exported decoded ONNX to: {onnx_path}.")

_split_onnx_for_dsp(onnx_path)


### Validating the ONNX output shapes

This cell checks that the exported ONNX has exactly two outputs — `bboxes` with shape `[1, 8400, 4]` and `scores` with shape `[1, 8400, 81]`. If you still see a single `detections` output with 85 columns, delete the ONNX file and re-run the previous cell so that `_split_onnx_for_dsp()` can restructure the graph.

In [ ]:
onnx_model = onnx.load("../models/LibreYOLOXs.onnx")
onnx.checker.check_model(onnx_model)

print("Inputs:")
for tensor in onnx_model.graph.input:
    dims = [d.dim_value if d.dim_value > 0 else d.dim_param for d in tensor.type.tensor_type.shape.dim]
    print(f" {tensor.name}: {dims}")

print("Outputs:")
output_shapes = {}
for tensor in onnx_model.graph.output:
    dims = [d.dim_value if d.dim_value > 0 else d.dim_param for d in tensor.type.tensor_type.shape.dim]
    output_shapes[tensor.name] = dims
    print(f" {tensor.name}: {dims}")

assert len(onnx_model.graph.output) == 2, "Expected exactly two outputs (bboxes, scores)."
assert "bboxes" in output_shapes, "Expected output named `bboxes`."
assert "scores" in output_shapes, "Expected output named `scores`."
assert output_shapes["bboxes"] == [1, 8400, 4], f"Unexpected bboxes shape: {output_shapes['bboxes']}"
assert output_shapes["scores"] == [1, 8400, 81], f"Unexpected scores shape: {output_shapes['scores']}"

print("ONNX export is correct: bboxes [1, 8400, 4] and scores [1, 8400, 81].")

## Converting the Model to QAIRT DLC Format

QAIRT uses the **DLC (Deep Learning Container)** format as an intermediate representation. The first step is to convert the ONNX model to a floating-point (**FP32**) DLC using the `qairt-converter` tool. This DLC file is the starting point for all subsequent quantization and compilation steps.

In [ ]:
# Keep the exported `images` input in its original NCHW layout.
!bash -c 'qairt-converter \
  --input_network ../models/LibreYOLOXs.onnx \
  --source_model_input_shape images 1,3,640,640 \
  --source_model_input_layout images NCHW \
  --output_path ../models/qairt/LibreYOLOXs_fp32.dlc'

### Inspecting the FP32 DLC

The `qairt-dlc-info` command prints a detailed summary of the DLC graph, including layer names, types, input/output tensor shapes, and supported execution backends (CPU, GPU, HTP). Reviewing this output verifies that the ONNX export was clean and that all operators are supported by QAIRT before proceeding to quantization.

The converter command now explicitly keeps the notebook-facing `images` input in its original NCHW shape `[1, 3, 640, 640]`. QAIRT may still insert an internal transpose to its native layout after that preserved app-facing input.

In [ ]:
!bash -c 'qairt-dlc-info \
    --input_dlc ../models/qairt/LibreYOLOXs_fp32.dlc'

### INT8 Post-Training Quantization

Running inference in **8-bit integer (INT8)** precision is significantly faster and more energy-efficient on Qualcomm® hardware than FP32 — with only a small accuracy trade-off. The `qairt-quantizer` tool applies **post-training quantization (PTQ)** by computing per-layer scale factors from the calibration `.raw` samples prepared earlier, producing a quantized DLC.

#### Why the ONNX graph is restructured before quantization

The original YOLOX export produces per-scale `[1, n, 85]` tensors that mix decoded bounding-box coordinates (range **0 – 640**) with sigmoid-activated scores (range **0 – 1**) in the same tensor. With default **INT8 per-tensor quantization** (256 levels), the quantizer sets a single scale to cover the full 0–640 range:

```
scale ≈ 640 / 255 ≈ 2.5  (units per step)
```

Any activation below ~1.25 rounds to INT8 `0`. Since all objectness and class-probability values lie in [0, 1], they **all collapse to zero** on DSP while CPU/GPU (float32/float16) remain unaffected.

The `_split_onnx_for_dsp()` function rewires the graph to eliminate all mixed-range intermediate tensors. The `[1, n, 81]` score tensors already exist in the graph as separate nodes (Slice outputs from the raw transposed features); the surgery routes them directly into their own cross-scale Concat without ever merging with bbox coordinates:

| Output   | Range | INT8 scale         | Values in [0, 1] map to |
|----------|-------|--------------------|-------------------------|
| `bboxes` | 0–640 | ≈ 2.5 / step       | INT8 levels 0–0 (lost)  |
| `scores` | 0–1   | ≈ 0.004 / step     | INT8 levels 0–255 ✓     |

This gives `scores` full INT8 resolution without widening any activations to INT16, preserving full DSP throughput.

In [ ]:
!bash -c 'qairt-quantizer \
    --input_dlc ../models/qairt/LibreYOLOXs_fp32.dlc \
    --input_list qairt/calib/filenames.txt \
    --output_dlc ../models/qairt/LibreYOLOXs_int8.dlc'

### Inspecting the INT8 DLC

After quantization, `qairt-dlc-info` is used again to inspect the INT8 DLC. Compared to the FP32 output, you should observe that layer types now reflect quantized variants. The INT8 DLC is expected to be smaller than the FP32 DLC, but the actual reduction should be measured.

As with the FP32 DLC, the committed notebook should preserve `images` as an NCHW input even if QAIRT uses an internal transpose before the native execution layout.

In [ ]:
!bash -c 'qairt-dlc-info \
    --input_dlc ../models/qairt/LibreYOLOXs_int8.dlc'

By following these steps, the model is optimized for efficient inference on Qualcomm® platforms, leveraging hardware acceleration for real-time AI applications. This process ensures that the model is both accurate and performant, making it suitable for deployment in edge devices powered by Qualcomm® chipsets.